In [31]:
import pandas as pd
import joblib
import numpy as np

# 加载数据
df = pd.read_excel('Target_data_500.xlsx')
qf_list = pd.read_excel('G:/AIMS_Lab/my_project/合金（下）/Small_Data_new/20240607FULL_final.xlsx', sheet_name='qf_21').columns.tolist()[:-2]
kl_list = pd.read_excel('G:/AIMS_Lab/my_project/合金（下）/Small_Data_new/20240607FULL_final.xlsx', sheet_name='kl_21').columns.tolist()[:-2]

qf_df = df[qf_list]
kl_df = df[kl_list]

# 预测并平均结果
def predict_average(model_paths, data):
    predictions = []
    for model_path in model_paths:
        model = joblib.load(model_path)
        pred = model.predict(data)
        predictions.append(pred)
    return np.mean(predictions, axis=0)

# 模型路径
qf_model_paths = [f'G:/AIMS_Lab/my_project/合金（下）/Small_Data_new/final_model/qf/rf/rf_{i}.pkl' for i in range(0, 5)]
kl_model_paths = [f'G:/AIMS_Lab/my_project/合金（下）/Small_Data_new/final_model/kl/rf//rf_{i}.pkl' for i in range(0, 5)]

# 进行预测
qf_predictions = predict_average(qf_model_paths, qf_df)
kl_predictions = predict_average(kl_model_paths, kl_df)

df=pd.read_excel('materials_data_combined_500.xlsx')
# 将预测结果添加到DataFrame中
df['屈服强度'] = qf_predictions
df['抗拉强度'] = kl_predictions

# 保存结果到新Excel文件
df.to_excel('Prediction_Results_760.xlsx', index=False)

In [29]:

import pandas as pd
import random
import re

# List of elements to include as columns
elements = ['Al', 'Zn', 'Mn', 'Mg', 'Li', 'Sn', 'Ce', 'La', 'Ca', 'Si', 'Cu', 'Ni',
            'Fe', 'Bi', 'Ti', 'Sr', 'Sm', 'Sc', 'Gd', 'Y', 'Zr', 'Nd', 'Ag', 'Yb']

material_names = ["Mg-11Y", "Mg-8Y-3Zn", "Mg-11Y-1Al", "Mg-8Gd-3Y-1Zn"]

# Initialize the DataFrame with zero values for each element and material column
comp_df = pd.DataFrame(0.0, index=range(len(material_names)), columns=elements + ['material'])

def extract_elements(compound):
    matches = re.findall(r'(\d+\.?\d*)\s*([A-Za-z]+)', compound)
    element_dict = {elem: float(pct) for pct, elem in matches}
    return element_dict

comp_df['material'] = material_names

# Apply extraction to each row
for i, material in enumerate(material_names):
    compound = re.sub(r'\s*\([^)]*\)', '', material)
    element_dict = extract_elements(compound)
    for element, percentage in element_dict.items():
        if element in elements:
            comp_df.at[i, element] = percentage
    total_other_elements = sum(element_dict.values())
    comp_df.at[i, 'Mg'] = 100.0 - total_other_elements

# Generate new features
materials = {
    "Mg-11Y": {"habit_planes": 5, "precipitate_distribution": 3},
    "Mg-8Y-3Zn": {"habit_planes": 4, "precipitate_distribution": 3},
    "Mg-11Y-1Al": {"habit_planes": 5, "precipitate_distribution": 3},
    "Mg-8Gd-3Y-1Zn": {"habit_planes": 5, "precipitate_distribution": 3}
}

data = []

for material in materials:
    for _ in range(400):
        crystal_size = random.choice([5, 10, 15, 20])
        habit_planes = 4 if material == "Mg-8Y-3Zn" else 5
        base_proportion = random.choice(range(15, 21))
        base_length = random.choice(range(5, 21, 5))
        base_diameter = random.choice(range(5, 21, 5))
        base_thickness = random.choice(range(2, 10))
        column_proportion = 0 if material == "Mg-8Y-3Zn" else random.choice(range(15, 21))
        column_length = 0 if material == "Mg-8Y-3Zn" else random.choice(range(20, 51, 10))
        column_diameter = 0 if material == "Mg-8Y-3Zn" else random.choice(range(20, 51, 10))
        column_thickness = 0 if material == "Mg-8Y-3Zn" else random.choice(range(5, 21, 5))

        features = {
            "Grain Size": crystal_size,
            "Habit Plane": habit_planes,
            "Distribution of precipitation": 3,
            "Fraction of basal phase": base_proportion,
            "Length of Basal phase": base_length,
            "Diameter of basal phase": base_diameter,
            "Width of basal phase": base_thickness,
            "Fraction of prismatic phase": column_proportion,
            "Length of prismatic phase": column_length,
            "Diameter of prismatic phase": column_diameter,
            "Width of prismatic phase": column_thickness
        }
        if base_diameter-crystal_size > 0 or base_length-crystal_size >0:
            continue
        data.append({
            "material": material,
            **features
        })

feature_df = pd.DataFrame(data)

# Merge original composition data with new feature data
merged_df = feature_df.merge(comp_df, on='material', how='left')
display(merged_df)
# Write to Excel
with pd.ExcelWriter('materials_data_combined_500.xlsx', engine='openpyxl') as writer:
    merged_df.to_excel(writer, index=False, sheet_name='CombinedData')

# 生成预测数据集结果
import pandas as pd
from matminer.featurizers.composition.composite import ElementProperty
from matminer.featurizers.composition.alloy import WenAlloys
from matminer.featurizers.composition.element import ElementFraction
from matminer.featurizers.composition.composite import Meredig
import pandas as pd
# 创建 DataFrame
from strength_calculation import calculate_Strength
from pymatgen.core import Composition


def to_comp(x):
    try:
        return Composition.from_weight_dict(x.to_dict())
    except Exception as ex:
        print(x, ex)

def main():
    # 对析出相和惯习面进行编码
    df = pd.read_excel('materials_data_combined_500.xlsx')

    # print(df)
    elements = ['Al', 'Zn', 'Mn', 'Mg', 'Li', 'Sn', 'Ce', 'La', 'Ca', 'Si', 'Cu', 'Ni',
                'Fe', 'Bi', 'Ti', 'Sr', 'Sm', 'Sc', 'Gd', 'Y', 'Zr', 'Nd', 'Ag', 'Yb']
    weightPercentage = df[elements]

    compSeries = weightPercentage.apply(to_comp, axis = 1)
    print(compSeries)

    df_comp = pd.DataFrame({"composition": compSeries})
    # print('hello')
    # 初始化特征扩展器
    featurizer = ElementProperty.from_preset("magpie")
    # 应用特征扩展器
    df_comp = featurizer.featurize_dataframe(df_comp, "composition")
    df_comp = WenAlloys().featurize_dataframe(df_comp, "composition")
    df_comp = Meredig().featurize_dataframe(df_comp, "composition")
    for columnName, obj in df_comp.dtypes.items():
        if obj == "object":
            print(columnName)
            del df_comp[columnName]

    all = pd.concat([df_comp, df], axis = 1)
    all = calculate_Strength(all)
    all.to_excel('Target_data_500.xlsx')

if __name__ == '__main__':
    main()

import pandas as pd
df = pd.read_excel('materials_data_combined_500.xlsx')
display(df)
# %pip install matminer,pymatgen

PermissionError: [Errno 13] Permission denied: 'Prediction_Results_365.xlsx'

In [46]:
import pandas as pd
df = pd.read_excel('Target_data.xlsx')
display(df)
qf_list = pd.read_excel('G:/AIMS_Lab/my_project/合金（下）/Small_Data_new/20240607FULL_final.xlsx',sheet_name='qf_21').columns.tolist()[:-2]
kl_list = pd.read_excel('G:/AIMS_Lab/my_project/合金（下）/Small_Data_new/20240607FULL_final.xlsx',sheet_name='kl_21').columns.tolist()[:-2]
print(qf_list)
# print(kl_list)
qf_df=df[qf_list]
kl_df=df[kl_list] 

# df_qf = df[]
# %pip install matminer,pymatgen

,material,MagpieData minimum Number,MagpieData maximum Number,MagpieData range Number,MagpieData mean Number,MagpieData avg_dev Number,MagpieData mode Number,MagpieData minimum MendeleevNumber,MagpieData maximum MendeleevNumber,MagpieData range MendeleevNumber,...,Sc,Gd,Y,Zr,Nd,Ag,Yb,Calculated Solid Solution,Calculated Grain Boundary,Calculated Yield Strength
0,Mg-11Y,12,39,27,12.882470,1.707255,12,12,68,56,...,0,0,11,0,0,0,0,81.780078,62.660761,159.440839
1,Mg-11Y,12,39,27,12.882470,1.707255,12,12,68,56,...,0,0,11,0,0,0,0,81.780078,62.660761,159.440839
2,Mg-11Y,12,39,27,12.882470,1.707255,12,12,68,56,...,0,0,11,0,0,0,0,81.780078,76.743445,173.523523
3,Mg-11Y,12,39,27,12.882470,1.707255,12,12,68,56,...,0,0,11,0,0,0,0,81.780078,76.743445,173.523523
4,Mg-11Y,12,39,27,12.882470,1.707255,12,12,68,56,...,0,0,11,0,0,0,0,81.780078,108.531621,205.311699
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,Mg-8Gd-3Y-1Zn,12,64,52,13.029882,2.004453,12,12,69,57,...,0,8,3,0,0,0,0,69.184112,80.812350,164.996462
116,Mg-8Gd-3Y-1Zn,12,64,52,13.029882,2.004453,12,12,69,57,...,0,8,3,0,0,0,0,69.184112,80.812350,164.996462
117,Mg-8Gd-3Y-1Zn,12,64,52,13.029882,2.004453,12,12,69,57,...,0,8,3,0,0,0,0,69.184112,80.812350,164.996462
118,Mg-8Gd-3Y-1Zn,12,64,52,13.029882,2.004453,12,12,69,57,...,0,8,3,0,0,0,0,69.184112,161.624700,245.808812


['MagpieData avg_dev MeltingT', 'Length of prismatic phase', 'mean AtomicRadius', 'Zr fraction', 'Fraction of prismatic phase', 'Width of prismatic phase', 'frac f valence electrons', 'Diameter of prismatic phase', 'Diameter of basal phase', 'Width of basal phase', 'Calculated Grain Boundary', 'Interant electrons', 'Calculated Yield Strength', 'Length of Basal phase', 'Habit Plane', 'Fraction of basal phase', 'Yang omega', 'Distribution of precipitation', 'Grain Size']
